# M3L3 E21 - E07 con Langfuse: ver una traza básica

Objetivo: tomar la anatomía mínima de LangGraph de E07 y agregar observabilidad con Langfuse.

Al final de este ejercicio deberás poder abrir Langfuse y decir:
- Esta fue la llamada al modelo.
- Este fue el input.
- Este fue el output.
- Este fue el modelo usado.
- Estos fueron los tokens y la latencia, si el proveedor los reporta.

Este ejercicio no busca hacer un sistema grande. Busca que la traza se vea claro.

## Paso 1 - Conectar OpenAI y Langfuse

Este bloque es lo mínimo para que una llamada al LLM aparezca en Langfuse.

Qué vamos a hacer:
- Instalar LangGraph, LangChain OpenAI y Langfuse.
- Cargar la API key de OpenAI con `getpass`.
- Cargar las keys de Langfuse con `getpass`.
- Configurar `LANGFUSE_BASE_URL` para US Cloud.
- Crear `CallbackHandler`, que es el puente entre LangChain y Langfuse.

No guardamos ninguna key dentro del notebook. Cada estudiante la pega en runtime.

In [ ]:
!pip install langgraph langchain langchain-openai langfuse -q

import os
from getpass import getpass
from typing import TypedDict

from langchain_openai import ChatOpenAI
from langfuse import Langfuse
from langfuse.langchain import CallbackHandler
from langgraph.graph import StateGraph, START, END

os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ").strip()
os.environ["LANGFUSE_PUBLIC_KEY"] = getpass("Langfuse Public Key: ").strip()
os.environ["LANGFUSE_SECRET_KEY"] = getpass("Langfuse Secret Key: ").strip()

# US Cloud. Si tu proyecto está en EU, usar: https://cloud.langfuse.com
langfuse_base_url = input("Langfuse Base URL [https://us.cloud.langfuse.com]: ").strip() or "https://us.cloud.langfuse.com"
langfuse_base_url = langfuse_base_url.rstrip("/")
os.environ["LANGFUSE_BASE_URL"] = langfuse_base_url
os.environ["LANGFUSE_HOST"] = langfuse_base_url  # compatibilidad con integraciones anteriores

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
langfuse = Langfuse()

if not langfuse.auth_check():
    raise RuntimeError("Langfuse no autenticó. Revisar keys y Base URL.")

langfuse_handler = CallbackHandler()


def invoke_llm(prompt: str):
    # Todas las llamadas que pasen por acá quedan trazadas en Langfuse.
    return llm.invoke(prompt, config={"callbacks": [langfuse_handler]})

print("OpenAI + Langfuse listos")

## Paso 2 - State mínimo

LangGraph mueve un estado entre nodos. En este caso el estado tiene tres campos:
- `name`: dato de entrada.
- `message`: respuesta generada por el LLM.
- `formatted`: versión final formateada.

Es el mismo concepto de E07, pero ahora la llamada al modelo queda observada en Langfuse.

In [2]:
class GreetState(TypedDict):
    name: str
    message: str
    formatted: str

print("State definido")

State definido


## Paso 3 - Dos nodos simples

`greet` es el nodo importante para Langfuse porque llama al LLM usando `invoke_llm(...)`.

`format_output` no llama al LLM. Solo transforma texto. En Langfuse deberás ver la llamada del modelo, no esta transformación local.

In [ ]:
def greet(state: GreetState) -> dict:
    response = invoke_llm(
        f"En una sola oración amigable y en español, saludá a {state['name']}. Respondé solo el saludo."
    )
    return {"message": response.content.strip()}


def format_output(state: GreetState) -> dict:
    return {"formatted": f">> {state['message']} <<"}

print("Nodos definidos")

## Paso 4 - Compilar el grafo

El grafo es lineal:

`START -> greet -> format_output -> END`

La única llamada al LLM ocurre dentro de `greet`.

In [4]:
graph = StateGraph(GreetState)
graph.add_node("greet", greet)
graph.add_node("format_output", format_output)

graph.add_edge(START, "greet")
graph.add_edge("greet", "format_output")
graph.add_edge("format_output", END)

app = graph.compile()
print("Grafo E21 compilado")

Grafo E21 compilado


## Paso 5 - Ejecutar y mirar Langfuse

Esta celda ejecuta una sola consulta. Después de correrla:

1. Abrí Langfuse.
2. Entrá al proyecto.
3. Andá a `Tracing`.
4. Abrí la traza más reciente.
5. Buscá la generation del modelo `gpt-4o-mini`.

Qué contar en clase: El usuario no ve Langfuse, pero nosotros como desarrolladores vemos qué prompt llegó, qué respondió el modelo, cuánto tardó y cuántos tokens consumió.

In [ ]:
result = app.invoke({"name": "estudiante", "message": "", "formatted": ""})

print("Mensaje LLM:", result["message"])
print("Formateado: ", result["formatted"])

# En notebooks conviene forzar el envío antes de ir a la UI de Langfuse.
langfuse.flush()
print("Listo: abrí Langfuse -> Tracing y buscá la traza más reciente")

## Cómo explicarlo

Frase corta para clase:

> Langfuse es como una caja negra visible para sistemas con LLM. El grafo ejecuta normal, pero cada llamada al modelo deja una evidencia: input, output, modelo, tokens, costo y latencia.

Lo importante de este ejercicio:
- LangGraph organiza el flujo.
- LangChain llama al modelo.
- Langfuse observa la llamada sin cambiar la lógica del grafo.